# PlantVillage -> Pull Kaggle Kernel -> Prepare -> Train -> Push to Hugging Face (Fixed)

Open runtime → Change runtime type → GPU before running cells. This fixed notebook runs the kernel pull, prepares the dataset, trains using the repo script, and pushes the model to the Hugging Face Hub.


In [ ]:
# Install required packages
!pip install -q transformers accelerate evaluate huggingface_hub torch torchvision timm kaggle papermill

## Upload Kaggle credentials (kaggle.json)
Either upload kaggle.json when prompted or set KAGGLE_USERNAME/KAGGLE_KEY as env vars.


In [ ]:
from google.colab import files
import os
print('If you prefer env vars, set KAGGLE_USERNAME and KAGGLE_KEY in the runtime before running.')
uploaded = files.upload()
if uploaded:
  os.makedirs('/root/.kaggle', exist_ok=True)
  for fn, data in uploaded.items():
    with open('/root/.kaggle/kaggle.json','wb') as f:
      f.write(data)
  os.chmod('/root/.kaggle/kaggle.json', 0o600)
  print('kaggle.json saved to /root/.kaggle/kaggle.json')


## Pull Kaggle kernel and inspect files
This will pull the specified kernel (imtkaggleteam/plant-diseases-detection-pytorch) into /content/kaggle_kernel.


In [ ]:
!kaggle kernels pull imtkaggleteam/plant-diseases-detection-pytorch -p /content/kaggle_kernel --unzip || true
!ls -la /content/kaggle_kernel || true


## Prepare PlantVillage dataset (download & create splits)
The repository contains a helper script scripts/hf_finetune/download_plantvillage_kaggle.py which will download and prepare the dataset.


In [ ]:
RAW_OUT='/content/plantvillage_raw'
HF_PREPARE_OUT='/content/hf_dataset'
!python scripts/hf_finetune/download_plantvillage_kaggle.py --dataset emmarex/plantdisease --out_dir $RAW_OUT --prepare_out $HF_PREPARE_OUT || true
!ls -la $HF_PREPARE_OUT || true


## Login to Hugging Face
Paste your Hugging Face token when prompted. This enables Trainer.push_to_hub to work.


In [ ]:
from getpass import getpass
from huggingface_hub import login
hf_token = getpass('Enter your Hugging Face token (will not be shown):')
if hf_token:
  login(token=hf_token)
  import os
  os.environ['HF_TOKEN'] = hf_token
  print('Logged in to Hugging Face')
else:
  print('No token provided — push_to_hub will not work')


## Train using repo script (or inspect pulled kernel to run its notebook)
Set HUB_MODEL_ID to the Hugging Face repo you want to push to.


In [ ]:
# Edit this: replace with your HF repo id (e.g. your-username/plant-vit)
HUB_MODEL_ID='your-username/plant-vit'
MODEL_NAME='google/vit-base-patch16-224'
DATA_DIR=HF_PREPARE_OUT
OUTPUT_DIR='/content/outputs/plant-vit'
!python scripts/hf_finetune/train.py --dataset_path $DATA_DIR --model_name_or_path $MODEL_NAME --output_dir $OUTPUT_DIR --per_device_train_batch_size 16 --num_train_epochs 6 --push_to_hub --hub_model_id $HUB_MODEL_ID || true


## After training
If training succeeded and push_to_hub worked, the model will be available at the specified HUB_MODEL_ID. Update HUGGINGFACE_MODEL on Netlify with that id and redeploy the site.
